# Assignment: Conservative NN

Conventional neural networks are pure pattern matchers: they fit whatever
function minimizes the training loss, with no notion of physical plausibility.
In many scientific and engineering domains, however, we know part of the
truth in advance (a conservation law, a symmetry, a monotonicity constraint)
and we would like the model to respect it *by construction*, not just
approximately after enough training data.

> For example, consider a river basin instrumented with several flow gauges. At each time step, the water flow measured at a downstream station should equal the combined upstream inflows, plus local rainfall contributions, minus losses due to evaporation, infiltration, or water extraction. A conventional neural network might predict the downstream flow directly from historical measurements and weather variables. Although its predictions may fit the observed data well, nothing prevents the model from implicitly creating or removing water, producing estimates that violate the conservation of mass. A conservative neural network, instead, predicts only the unknown local gains and losses, while the downstream flow is computed using the mass balance equation. As a result, every prediction automatically satisfies the conservation law while still allowing the model to learn the complex hydrological processes that cannot be described analytically.

In this assignment you will build a small but complete example of this idea:
a **water reservoir** whose mass balance is governed by a known conservation
law. You will train two recurrent models on the same data and the same loss
function:

1. A **conventional LSTM** (baseline, Option 1) that predicts the next mass value directly.
2. **Your chosen architecture** (Option 2, 3, or 4 from §2) that incorporates the conservation law.

Both are functions of the same inputs, trained with the same MSE loss and the
same optimizer. The only difference is architectural: one has the
conservation law baked into its output layer, the other does not. Comparing
them lets you measure, empirically, what a **physical constraint** buys
you over a network that has to *learn* the constraint from data alone.

The implementation will be carried out in **JAX** (with **Flax** for the
neural network layers and **Optax** for optimization).

### Learning objectives

By the end of this assignment you should be able to:

- Explain different ways to use physical constraints in neural networks.
- Implement, train, and debug a recurrent model in JAX/Flax.
- Design an architecture that provably satisfies a known physical law.
- Quantitatively evaluate model quality along more than one axis (accuracy
  *and* physical consistency).
- Reason about physical knowledge as a form of **inductive bias**, and its
  effect on data efficiency.

## 1. Assignment Structure

This assignment runs in two phases.

### Phase A — In-class group exercise (no code)

**Goal**: understand the problem *before* writing any code.

In small groups, discuss and produce a short written plan (a few bullet
points per item is enough) covering:

1. **Design a specific scenario.** Draw the reservoir system you will consider: what flows in, what flows out, what is measured, what is hidden. Write down the conservation
   equation in your own words.
2. **Why a plain LSTM can fail.** Give a concrete example of a prediction a
   conventional LSTM could make that would be physically impossible in this
   system.
3. **The solution space.** There are different ways you could make a
   neural network respect the conservation law (see below). For each alternative, discuss:
    1. Which approaches guarantee conservation?
    2. Which are easiest to implement?
    3. Which are likely to generalize best with limited data?
    4. Which would you choose for this scenario? Justify your answer.
4. **A plan.** Sketch, at a high level, the architecture you will implement
   at home, and the metrics you will use to judge whether it worked.

Bring this plan to the individual phase — you will implement it.

### Phase B — Individual take-home implementation

Goal: implement, train, evaluate, and reflect. See
`A1_Phase_B_Individual_Exercise.ipynb` for the full task breakdown and
deliverables checklist.


# Phase A — Classroom Group Exercise

## Conservative NN architectures: Understanding the Problem

**Format:** small groups, in class, **no coding**.

**Goal:** understand the physical scenario and the design problem *before* writing any code. What your group produces here is the plan you will implement individually in Phase B (`A1_Phase_B_Individual_Exercise.ipynb`).

Fill in your group's answers directly in the empty markdown cells below (double-click a cell to edit it, then run it to render).

## Objectives of this session

- Understand why a conventional neural network can produce physically impossible predictions.
- Explore the water-reservoir scenario and its conservation law.
- Compare candidate solution strategies for enforcing the law.
- Produce a short written plan — architecture + metrics — for the individual implementation in Phase B.

## 1. The scenario: a water reservoir

We model a water reservoir observed over discrete time steps $t = 0, 1, 2, ...$. At each step:

| Symbol | Meaning | Observed? |
|---|---|---|
| $m(t)$ | Water mass (volume) in the reservoir | yes |
| $r(t)$ | Inflow (e.g. rainfall, pumping in) | yes |
| $q(t)$ | Controlled outflow (e.g. a valve setting) | yes |
| $c(t)$ | Hidden consumption (e.g. evaporation, leakage, unmetered usage) | **no** |

The system evolves according to an exact mass balance:

$$
m(t+1) = m(t) + r(t) - q(t) - c(t)
$$

$c(t)$ is *not* directly observable. It depends on temperature, time of day, day of the year and a small random component, and must be inferred from context. It is exactly the term a model would have to estimate.

## 2. Physical knowledge and inductive bias

There is more than one way to incorporate the conservation law into a neural network. Each way implement a different **inductive bias** in the network. 

Discuss the following alternatives and decide which one is most appropriate for this problem. For each option, consider its advantages and disadvantages in terms of **physical consistency, ease of training, interpretability, and generalization**.

### Option 1. Direct Prediction (Baseline - no conservation law)

The network (a conventional LSTM) directly predicts the next water mass:

$$\hat m(t+1)=\mathrm{LSTM}(x(t))$$

At each time step, the input vector $x(t)$ contains the current water mass $m(t)$, the inflow $r(t)$, the controlled outflow $q(t)$, and any other available variables such as temperature , time of day, and day of the year. However, the system is also influenced by hidden factors that are not directly measured, including evaporation, soil infiltration, small leaks, unreported water withdrawals, and other environmental effects. These hidden processes determine an unknown water loss $c(t)$, which cannot be observed directly but must be inferred from the available measurements.

No conservation law is enforced. The model is free to learn any mapping that minimizes the prediction error.

Advantages

* Simplest architecture.
* Maximum flexibility.

Disadvantages

* Predictions may violate the conservation law.
* Requires the network to learn physical principles from data alone.

### Option 2. Conservation as a Soft Constraint

The network has two outputs: it predicts both the next water mass and the estimated water consumption,

$$(\hat m(t+1),\,\hat c(t)) = \mathrm{LSTM}(x(t)).$$

The prediction loss is combined with an additional penalty that measures how well the predicted quantities satisfy the conservation law:

$$L = L_{\text{prediction}} + \lambda L_{\text{physics}},$$

where

$$L_{\text{prediction}}
=
\big(\hat m(t+1)-m(t+1)\big)^2,$$

and

$$L_{\text{physics}}
=
\left(
\hat m(t+1)-m(t)-r(t)+q(t)+\hat c(t)
\right)^2.
$$

The parameter $\lambda$ controls the trade-off between prediction accuracy and adherence to the conservation law.

Advantages

* Easy to incorporate into an existing neural network.
* Encourages the model to learn physically consistent predictions.
* Allows some flexibility when the governing equation is only approximately valid or when measurements are noisy.

Disadvantages

* Conservation is only encouraged, not guaranteed.
* The quality of the solution depends on the choice of the penalty weight \lambda.
* The model must learn two related quantities simultaneously, which may make training more difficult and slower to converge.

### Option 3. Conservative Architecture (Hard Constraint)

The network predicts only the unknown hidden consumption:

$$\hat c(t)=\mathrm{LSTM}(x(t))$$

The final water mass is computed analytically:

$$\hat m(t+1)=m(t)+r(t)-q(t)-\hat c(t).$$

Advantages

* Conservation is satisfied exactly.
* The network focuses only on learning the unknown process.
* Strong inductive bias can improve generalization.

Disadvantages

* Requires that the governing equation is known.
* Less flexible if the physical model is approximate.
  
### Option 4. Predict Residual Corrections

Suppose the conservation equation is only approximately correct. The network predicts a small correction:

$$\hat m(t+1) = m(t)+r(t)-q(t)-\hat c(t) +\delta(t), $$

where the network learns both the hidden consumption and the correction term.

Advantages

* Can compensate for modeling errors.
* Useful when the physics is only approximate.

Disadvantages

* The correction may absorb physical inconsistencies.
* Conservation is no longer exact.

The goal is not simply to obtain the lowest prediction error, but to understand how prior scientific knowledge can be integrated into neural network design, and how different design choices trade off flexibility, interpretability, and physical consistency. Discuss these four options as a group in Discussion 3 below.

## Discussion 1 — Draw the system

As a group, sketch the reservoir system: what flows in, what flows out, what is measured, what is hidden. Write the conservation equation in your own words (not just the symbols — explain what it means physically).

You can draw an ASCII/text diagram here, or describe it in words.

_Group answer — Discussion 1:_

*(write here)*

## Discussion 2 — Why can a plain LSTM fail?

Give a **concrete example** of a prediction a conventional LSTM (trained to predict `m(t+1)` directly from `[m(t), r(t), q(t), temp(t), hour(t), day_year(t)]`, with an MSE loss) could plausibly make that would be **physically impossible** in this system.

_Group answer — Discussion 2:_

*(write here)*

## Discussion 3 — Choose a solution strategy

Using the four options from §2, discuss as a group:

1. Which approaches guarantee conservation?
2. Which are easiest to implement?
3. Which are likely to generalize best with limited data?
4. Which would you choose for this scenario? Justify your answer.

_Group answer — Discussion 3:_

*(write here)*

## Discussion 4 — Your plan

Sketch, at a high level, the architecture your will (individualy) implement at home (Phase B), and the metrics you will use to judge whether it worked. This becomes your common plan for the individual implementation. Bring it with you.

Consider:
- What does the network predict, and what (if anything) is computed analytically afterward?
- What inputs does each part of the model need — and critically, should the true hidden consumption `c(t)` ever be a model input? Why or why not?
- How will you measure success? (Think about more than one axis — e.g. prediction accuracy *and* physical consistency.)

_Group answer — Discussion 4 (your plan):_

*(write here)*

---

## Handoff to Phase B

Open `A1_Phase_B_Individual_Exercise.ipynb` and work through it using the plan you just wrote: you'll build a **conventional LSTM** as your baseline and **your chosen architecture** (Option 2, 3, or 4 — whichever you sketched in Discussion 4), then compare them experimentally.